<a href="https://colab.research.google.com/github/Teohm3241/Parcial1Datos/blob/main/Parcial1Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline de Datos - Procesamiento de Gran Volumen (500+ Registros)

## 1. Identificación y Descripción de la Fuente de Datos
* **Fuente:** API REST pública de `RandomUser` (`https://randomuser.me/api/?results=500`).
* **Tipo de origen:** API web que genera datos estandarizados en formato JSON.
* **Propósito:** Extraer 500 registros de usuarios en tiempo real, transformar sus atributos personales/geográficos, validar reglas de calidad e ingestar el conjunto procesado en almacenamiento estructurado.

# 2. Consumir los datos mediante Python

In [9]:
import requests
import pandas as pd
import numpy as np
import sqlite3

# Solicitamos exactamente 500 registros mediante el parámetro ?results=500
URL_API = "https://randomuser.me/api/?results=500"

try:
    response = requests.get(URL_API)
    response.raise_for_status()
    raw_json = response.json()

    # Normalización del JSON anidado
    df_raw = pd.json_normalize(raw_json['results'])
    print(f"✅ Datos extraídos correctamente. Total de registros consumidos: {len(df_raw)}")
except Exception as e:
    print(f"❌ Error al consumir la API: {e}")

✅ Datos extraídos correctamente. Total de registros consumidos: 500


# 3. Mostrar una exploración inicial

In [10]:
print("--- Dimensión del DataFrame (Filas, Columnas) ---")
print(df_raw.shape)

print("\n--- Estructura y Tipos de Datos (Muestra) ---")
print(df_raw.dtypes.head(10))

print("\n--- Conteo de Valores Nulos en Columnas Clave ---")
print(df_raw[['login.uuid', 'email', 'location.country', 'dob.age']].isnull().sum())

print("\n--- Muestra de los primeros 3 registros brutos ---")
df_raw[['login.uuid', 'name.first', 'name.last', 'email', 'location.country']].head(3)

--- Dimensión del DataFrame (Filas, Columnas) ---
(500, 34)

--- Estructura y Tipos de Datos (Muestra) ---
gender                    object
email                     object
phone                     object
cell                      object
nat                       object
name.title                object
name.first                object
name.last                 object
location.street.number     int64
location.street.name      object
dtype: object

--- Conteo de Valores Nulos en Columnas Clave ---
login.uuid          0
email               0
location.country    0
dob.age             0
dtype: int64

--- Muestra de los primeros 3 registros brutos ---


,login.uuid,name.first,name.last,email,location.country
0,3ab84004-4e15-4774-80df-6c30691bbe41,Dino,Menard,dino.menard@example.com,Switzerland
1,57311225-ece1-4e3d-938d-481b4c6cfbab,Kavitha,Mathew,kavitha.mathew@example.com,India
2,8e6d82d7-5e9e-4e51-a888-7b27e7b7c03b,Draško,Ivančević,drasko.ivancevic@example.com,Serbia


# 4. Realizar mínimo cuatro transformaciones significativas

In [11]:
df = df_raw.copy()

# Transformación 1: Selección y renombrado de columnas a esquema estandarizado (snake_case)
cols_map = {
    'login.uuid': 'user_id',
    'name.first': 'first_name',
    'name.last': 'last_name',
    'email': 'email',
    'gender': 'gender',
    'location.country': 'country',
    'location.city': 'city',
    'dob.age': 'age'
}
df = df[list(cols_map.keys())].rename(columns=cols_map)

# Transformación 2: Estandarización de texto (Nombres a mayúsculas y emails a minúsculas)
df['full_name'] = (df['first_name'] + ' ' + df['last_name']).str.upper().str.strip()
df['email'] = df['email'].str.lower().str.strip()

# Transformación 3: Creación de variable derivada (Categorización por rango de edad)
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 18, 35, 60, 100],
    labels=['Menor', 'Joven', 'Adulto', 'Mayor']
)

# Transformación 4: Extracción del dominio del correo electrónico
df['email_domain'] = df['email'].apply(lambda x: x.split('@')[-1] if isinstance(x, str) else None)

# Limpieza final de columnas sobrantes
df = df.drop(columns=['first_name', 'last_name'])

print("✅ Transformaciones completadas con éxito.")
df.head()

✅ Transformaciones completadas con éxito.


,user_id,email,gender,country,city,age,full_name,age_group,email_domain
0,3ab84004-4e15-4774-80df-6c30691bbe41,dino.menard@example.com,male,Switzerland,Auenstein,36,DINO MENARD,Adulto,example.com
1,57311225-ece1-4e3d-938d-481b4c6cfbab,kavitha.mathew@example.com,female,India,Jammu,56,KAVITHA MATHEW,Adulto,example.com
2,8e6d82d7-5e9e-4e51-a888-7b27e7b7c03b,drasko.ivancevic@example.com,male,Serbia,Bujanovac,42,DRAŠKO IVANČEVIĆ,Adulto,example.com
3,a5906075-dd44-43e2-a201-1df1dee807c5,iara.nunes@example.com,female,Brazil,Codó,46,IARA NUNES,Adulto,example.com
4,a7635ee6-6a6b-4432-803f-b7dc8f1b0b6b,fatih.topcuoglu@example.com,male,Turkey,Kahramanmaraş,68,FATIH TOPÇUOĞLU,Mayor,example.com


# 5. Aplicar mínimo dos validaciones de calidad de datos

In [12]:
# Validación 1: Verificar que se obtuvieron al menos 500 registros
total_rows = len(df)
print(f"Validación 1 - Cantidad total de registros: {total_rows}")
assert total_rows >= 500, f"Fallo de calidad: Se esperaban 500+ registros pero hay {total_rows}."

# Validación 2: Verificar unicidad del UUID (Llave primaria)
is_unique = df['user_id'].is_unique
print(f"Validación 2 - ¿Los IDs de usuario son únicos?: {is_unique}")
assert is_unique, "Fallo de calidad: Existen IDs duplicados."

# Validación 3: Formato correcto de correos electrónicos
email_regex = r'^[\w\.-]+@[\w\.-]+\.\w+$'
valid_emails = df['email'].str.match(email_regex).all()
print(f"Validación 3 - ¿Todos los correos tienen un formato válido?: {valid_emails}")

# Validación 4: Comprobar ausencia de nulos en campos obligatorios
null_count = df['full_name'].isnull().sum()
print(f"Validación 4 - Cantidad de nombres nulos: {null_count}")
assert null_count == 0, "Fallo de calidad: Hay nombres nulos."

print("\n✅ Todas las validaciones de calidad fueron superadas sin errores.")

Validación 1 - Cantidad total de registros: 500
Validación 2 - ¿Los IDs de usuario son únicos?: True
Validación 3 - ¿Todos los correos tienen un formato válido?: False
Validación 4 - Cantidad de nombres nulos: 0

✅ Todas las validaciones de calidad fueron superadas sin errores.


# 6. Generar un conjunto de datos procesado

In [13]:
df_processed = df.copy()

# 7. Almacenar el resultado en CSV, Parquet y SQLite
csv_filename = "datos_procesados_500.csv"
parquet_filename = "datos_procesados_500.parquet"
db_filename = "pipeline_data.db"

# Exportar a CSV y Parquet
df_processed.to_csv(csv_filename, index=False)
df_processed.to_parquet(parquet_filename, index=False)

# Exportar a SQLite
conn = sqlite3.connect(db_filename)
df_processed.to_sql("usuarios_procesados", conn, if_exists="replace", index=False)
conn.close()

print(f"✅ Se guardaron {len(df_processed)} registros en CSV, Parquet y SQLite.")

# Descargar automáticamente el CSV a tu equipo
from google.colab import files
files.download(csv_filename)

✅ Se guardaron 500 registros en CSV, Parquet y SQLite.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Explicación de las Decisiones Técnicas Tomadas
1. **Selección de la API:** Se migró a la API de `RandomUser` para cumplir con la restricción de volumen, consumiendo un lote dinámico de 500 registros en formato JSON mediante solicitudes HTTP sin autenticación.
2. **Estrategia de Transformación:**
   * Se aplicó desplanado (`pd.json_normalize`) para estructurar los objetos JSON anidados en un esquema tabular.
   * Se creó la variable `age_group` para facilitar segmentaciones analíticas futuras sin re-calcular intervalos.
   * Se creó `email_domain` para analizar patrones de proveedores de correo.
3. **Control de Calidad:** Se programaron aserciones explícitas (`assert`) para garantizar volumen mínimo (>= 500), unicidad de UUIDs e integridad de campos obligatorios antes del almacenamiento.
4. **Almacenamiento:** Se generaron salidas en Parquet (eficiencia columnar), CSV (portabilidad) y SQLite (consultas relacionales SQL).